## DAY 1 - OPENAI API

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from google import genai
from IPython.display import Markdown, display, update_display

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')
g_api_key = os.getenv('GOOGLE_API_KEY')

In [ ]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
openai = OpenAI(base_url=GEMINI_BASE_URL,api_key=g_api_key)
MODEL = "gemini-3.1-flash-lite"

In [ ]:
system_prompt = """You are an helpful assitant who help users to explain things clearly with neat examples. 
Analyse the problem and breakdown and breielfy explain the topics to the user. 
Don't provide the answer in Markdown
"""
user_prompt = """
    What is Quantization in AI?
"""

# Step 2: Make the messages list

messages = [{
    "role": "system", "content": system_prompt,
    "role": "user", "content": user_prompt
}]

# Step 3: Call OpenAI
response = openai.chat.completions.create(
    model="gemini-3.1-flash-lite",
    messages=messages
)

# Step 4: print the result
display(Markdown(response.choices[0].message.content))

## Day 4 - Tokens and Tokenizer

In [ ]:
# Import tiktoken package from openai to validate how the tokens are splitted
import tiktoken

encoding = tiktoken.encoding_for_model("gpt-4")

In [ ]:
# Encode the words into tokens using encoding.encode
tokens = encoding.encode("Convert this sentence into tokens which can be feeded into LLM as the input.")
tokens

In [ ]:
# Display the unique tokens and their token_text mappings

for token in tokens:
    token_text = encoding.decode([token])
    print(f"{token}: {token_text}")

In [ ]:
# Validate the tokens for different words
print(encoding.decode([279]))
print(encoding.decode([55667]))

### Brochure provider for a website

In [ ]:
from scraper import fetch_website_links, fetch_website_contents
import json

In [ ]:
# System Prompt for getting the relevant

link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""


In [ ]:
# This has the user_prompt along with the scrapped links

def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [ ]:
# API call to get the relevant links from the list of linsk from the website
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [ ]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [ ]:
# Uses both the fetch content and website linsk from the scrapper and append it in a variable

def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [ ]:
# Creating a structired user prompt for the creating brochure based on content and links

def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [ ]:
# providing the website related input which we got earlier and feeding inot different llm model and streaming the output

def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gemini-3.8-flash",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
# Display the streamed results.

stream_brochure("HuggingFace", "https://huggingface.co")